# The story: every candidate slide figure, in order

One section per question, in the order the analysis folders answer them. Each section has the question, the figure, the key numbers (printed from the results, so they stay current), and a draft takeaway.

This notebook draws no figures of its own: it shows the PNGs the folder notebooks save (see `cc_stats.plots.save_figure`), and prints its numbers from the CSVs the folder scripts write to `results/<version>/`, so it agrees with the folders by construction. Run `pixi run run-analysis` first; `run_all.py` runs this notebook last.

**The draft takeaways quote numbers checked on v2 on 2026-10-20.** Re-check them after a rebuild.

Every other figure is described in `figure_catalogue.ipynb`.

Engel is the primary outcome. Each outcome section also shows the **ILAE** version (improved = ILAE 1-4, at least a 50% reduction in seizure days), drawn by the folder notebooks, and the takeaway says whether ILAE agrees.

In [ ]:
import pandas as pd

from cc_stats.build_analysis import CORE_TYPES
from cc_stats.paths import ANALYSIS_CSV, DATASET, results_dir
from IPython.display import Image, display


print(f'dataset {DATASET}')
analysis = pd.read_csv(ANALYSIS_CSV, index_col='subject_id')
firth_all = pd.read_csv(results_dir('1_improvement/types') / 'firths_results.csv')
firth = firth_all.query("outcome == 'engel'")
disappearance = pd.read_csv(results_dir('2_disappearance/types') / 'disappearance_rates.csv').set_index('type')
improvement = pd.read_csv(results_dir('2_disappearance/types') / 'improvement_rates.csv')
by_change = pd.read_csv(results_dir('2_disappearance/types') / 'improvement_by_count_change.csv').query("outcome == 'engel'")
pairs = pd.read_csv(results_dir('4_combinations/types') / 'conditional_disappearance.csv')
combos = pd.read_csv(results_dir('4_combinations/types') / 'combinations_and_outcome.csv')


def fmt_p(v):
    return '<0.001' if v < 0.001 else f'{v:.3f}'


def frac(k, n):
    return f'{int(k)}/{int(n)} ({k / n:.0%})'


# A figure a folder notebook saved (see cc_stats.plots.save_figure).
def show(folder, name):
    path = results_dir(folder) / 'figures' / f'{name}.png'
    if not path.exists():
        raise FileNotFoundError(f'{path} not found - run `pixi run run-analysis` first')
    display(Image(filename=path, width=700))

## 0. Who are the patients?

In [ ]:
show('0_cohort', 'patient_flow')

recorded = analysis[analysis['engel_class'].notna()]
print(f"Engel improved (I-III): {frac((recorded['engel_improved'] == 1).sum(), len(recorded))}")
print(f"LGS: {frac((analysis['lgs'] == 1).sum(), analysis['lgs'].notna().sum())}")
age = analysis['age_at_cc'].dropna()
print(f"Age at surgery: median {age.median():.0f} years (IQR {age.quantile(.25):.0f}-{age.quantile(.75):.0f}, n={len(age)})")

Most of "improved" is Engel III: worthwhile improvement, but not seizure freedom. From `0_cohort/outcome_overview.ipynb`.

In [ ]:
show('0_cohort', 'outcome_class_distribution_engel')

**ILAE:** most improved patients are ILAE 4 (at least a 50% reduction in seizure days); 74% improved.

In [ ]:
show('0_cohort', 'outcome_class_distribution_ilae')

## 1. Which pre-op features go with improvement?

The share of patients who improved with vs without each feature before surgery, with Wilson 95% CIs. From `1_improvement/types/Firth_results.ipynb`.

In [ ]:
show('1_improvement/types', 'observed_improvement_with_without_engel')

for name in ('myoclonic', 'lgs'):
    r = firth.set_index('predictor').loc[name]
    print(f"{name}: improved {frac(r['improved_with'], r['n_with'])} with vs {frac(r['improved_without'], r['n_without'])} "
          f"without; Firth OR {r['OR']:.2f} ({r['lo']:.2f}-{r['hi']:.2f}), p={fmt_p(r['p'])}, Holm {fmt_p(r['p_holm'])}")

The same comparison as Firth odds ratios, each seizure type adjusted for LGS (two-predictor models, 8 events per predictor on Engel, 11.5 on ILAE), Engel (teal) and ILAE (orange) together, with group sizes and Holm p. The unadjusted ORs are nearly identical (`1_improvement/types/Firth_results.ipynb` 2a, and the catalogue).

In [ ]:
show('1_improvement/types', 'firth_types_lgs_adjusted')

Patient characteristics: observed improvement by sex, both outcomes, with Fisher's exact p. From `0_cohort/patient_characteristics.ipynb`.

In [ ]:
show('0_cohort', 'observed_improvement_by_sex')

**ILAE:** the with-vs-without figure on ILAE (n = 88, 23 not improved). The adjusted ORs above already show both outcomes.

In [ ]:
show('1_improvement/types', 'observed_improvement_with_without_ilae')

**Draft takeaway:** Patients with myoclonic seizures before surgery almost all improved: 28/29 (97%) vs 51/66 (77%) without, Firth OR 5.7 (1.3-54). It's the only feature that stands out; on Engel it doesn't survive correction for the 11 features tested (Holm 0.19). LGS made no difference (82% vs 84%, OR 0.88), and adjusting for LGS doesn't change myoclonic's OR. **ILAE agrees and is clearer:** 93% vs 66%, OR 5.4 (1.6-29), p = 0.006 (Holm 0.07); adjusted for LGS, OR 6.1 with Holm p = 0.039, the only corrected p below 0.05 among the seizure types (by single code, generalized myoclonic, code 5, also survives on ILAE: Holm p 0.037, 0.018 adjusted for LGS). On ILAE, LGS leans toward worse outcomes (67% vs 80%, OR 0.53), as in the reference deck, but isn't significant.

## 1b. Does the myoclonic advantage hold within subgroups?

Among patients with each other core type (or LGS), improvement with vs without the feature: the six strongest pairs of 30. From `4_combinations/types/combinations_and_outcome.ipynb`.

In [ ]:
show('4_combinations/types', 'combinations_and_outcome_strongest_engel')
show('4_combinations/types', 'combinations_and_outcome_strongest_ilae')

for outcome in ('engel', 'ilae'):
    r = combos.query("outcome == @outcome and feature == 'myoclonic' and context == 'lgs'").iloc[0]
    print(f"{outcome}, among LGS: myoclonic {frac(r['improved_with'], r['n_with'])} vs "
          f"{frac(r['improved_without'], r['n_without'])} without; p={fmt_p(r['p'])}, Holm {fmt_p(r['p_holm'])}")

**Draft takeaway:** Myoclonic patients improve more whatever else they have. Among LGS patients, all 16 with myoclonic seizures improved vs 21/29 (72%) without (Engel); on ILAE 14/15 (93%) vs 15/28 (54%), the strongest of the 30 pairs. The same holds among patients with BTC and with atonic seizures. Among LGS patients who also have a tonic code (a stricter LGS group), the pattern holds (Engel 12/12 vs 16/21, ILAE 11/12 vs 11/20). Nothing survives correction across the 30 pairs, so this supports the myoclonic finding rather than adding a new one. LGS patients *without* myoclonic are the worst-off group (54% improved on ILAE), which is where the reference deck's "LGS is bad" comes from.

## 2. How often does each seizure type disappear?

Among patients who had the type before surgery and whose post-op types were recorded. Wilson 95% CIs.

In [ ]:
show('2_disappearance/types', 'disappearance_rate_core_types')

**Draft takeaway:** Seizure types differ a lot in how often they disappear. Absence and atonic disappeared in about 45% of patients who had them, while tonic disappeared in about 12% and BTC in about 21%.

## 3. Does disappearance go with improvement?

Engel improvement among patients whose type **left** vs **stayed** after surgery.

In [ ]:
show('2_disappearance/types', 'improvement_stayed_vs_left_core_types_engel')

d = improvement[(improvement['outcome'] == 'engel')].set_index('type').loc[CORE_TYPES]
for t, r in d.iterrows():
    print(f"{t}: left {frac(r['left_improved'], r['left_n'])} vs stayed {frac(r['stayed_improved'], r['stayed_n'])}")

**ILAE:** the same on ILAE.

In [ ]:
show('2_disappearance/types', 'improvement_stayed_vs_left_core_types_ilae')

**Draft takeaway:** For atonic, BTC, tonic and absence, patients whose type disappeared improved more often (by 10-18 points). But most patients improved even when the type stayed (71-100%), and the CIs overlap throughout: no difference survives correction (Fisher's exact test, Holm across 10 groups; smallest Holm p 0.68). Myoclonic is the exception: all 18 patients whose myoclonic seizures persisted still improved. Disappearance goes with improvement only partly.

## 4. Does losing more types go with a better outcome?

Change in the number of core seizure types (0-5) from before to after surgery.

In [ ]:
show('2_disappearance/types', 'improvement_by_count_change_engel')

**ILAE:** improvement by change in type count.

In [ ]:
show('2_disappearance/types', 'improvement_by_count_change_ilae')

**Draft takeaway:** There's no steady "more types lost, better outcome" trend (Spearman rho -0.10 with Engel class). Everyone who lost 2+ types improved (13/13), but so did everyone who gained one (9/9). All 16 patients who didn't improve kept the same number of types or lost one. **ILAE disagrees:** on ILAE, losing more types does go with a better outcome (rho -0.26, p = 0.019; improved 65% with no change, 90% after losing one type, 100% after losing two or more). ILAE counts seizure days, so it tracks the loss of seizure types more closely than Engel. **Read this as description, not explanation:** losing seizure types is part of what both scales measure, and patients with more types have more to lose.

## 5. Combinations: does one seizure type change whether another disappears, or whether patients improve?

Each cell: the row's outcome with vs without the column's feature before surgery, in percentage points. First disappearance, then improvement (the grid behind section 1b). Holm-corrected within each grid.

In [ ]:
show('4_combinations/types', 'conditional_disappearance_heatmap')
show('4_combinations/types', 'combinations_and_outcome_heatmap_engel')

for _, r in pairs.sort_values('p').head(3).iterrows():
    print(f"{r['target']} disappears with {r['condition']}: {frac(r['left_with'], r['n_with'])} vs "
          f"{frac(r['left_without'], r['n_without'])} without; p={fmt_p(r['p'])}, Holm {fmt_p(r['p_holm'])}")

A simpler slide than the heatmap: the strongest pair, beside BTC by atonic (the reference deck's example). From `4_combinations/types/conditional_disappearance.ipynb`.

In [ ]:
show('4_combinations/types', 'strongest_pair_vs_btc_by_atonic')

**Draft takeaway:** No pair survives correction in either grid, so these are leads, not findings. Myoclonic stands out in both: BTC disappeared in 42% of patients who also had myoclonic seizures vs 12% without, and atonic in 73% vs 32%; and myoclonic's row in the improvement grid is positive in every column. LGS went the other way for tonic disappearance (3% vs 24%). The reference deck's example, BTC disappearing more often with atonic (70% vs 24% on v1), is 32% vs 15% here (p = 0.19).

## Candidate stories (15-minute talk)

Both use the figures above. At about 1.5 minutes a slide, 15 minutes is roughly 9-10 slides: title, background and cohort, then one slide per question, then conclusions.

**A. The reference deck's arc, answered with these data.**
Seizure types disappear at very different rates (section 2). Disappearance only partly tracks improvement (3), and losing more types doesn't steadily help on Engel, though it does on ILAE (4). Myoclonic is the one baseline feature that stands out (1), and LGS on its own doesn't predict outcome. So "what disappeared" explains only some of who benefits. Next: EEG, seizure frequency and more patients.

**B. "Most patients benefit, and myoclonic is the stand-out profile."** (recommended)
1. *Cohort:* 102 with Engel; 82% improved, mostly Engel III (section 0).
2. *Who benefits:* myoclonic patients almost all improved; LGS alone doesn't predict outcome (1).
3. *Is it just LGS?* No: adjusting for LGS leaves myoclonic's effect unchanged, and among LGS patients the myoclonic ones all improved (1, 1b). ILAE's adjusted Holm p = 0.039.
4. *Subgroups:* the myoclonic advantage holds among BTC and atonic patients too (1b, 5).
5. *What goes away:* types disappear at very different rates; myoclonic patients' other types disappear more often (2, 5).
6. *Does disappearance explain benefit?* Only partly: most improved even when their type stayed, and every patient whose myoclonic seizures persisted still improved (3, 4).
7. *Limits:* single coder, small non-improved group (16 Engel / 23 ILAE), no correction-surviving result on Engel, no EEG or seizure frequency.
8. *Next:* characterise the myoclonic profile (EEG, syndrome, e.g. whether any are myoclonic-atonic epilepsy), seizure frequency, more patients.